# Universe EDA
Explore the ticker universe from Polygon's `list_tickers` endpoint.

In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from polygon import RESTClient

load_dotenv()
client = RESTClient(api_key=os.environ['POLYGON_API_KEY'])

In [2]:
# Fetch all active US common stocks across NYSE + Nasdaq
raw = []
for exchange in ['XNYS', 'XNAS']:
    for t in client.list_tickers(
        market='stocks',
        exchange=exchange,
        type='CS',
        active=True,
        limit=1000,
    ):
        raw.append(vars(t))

df = pd.DataFrame(raw)
print(f'{len(df)} tickers fetched')
df.head()

5028 tickers fetched


,ticker,name,market,locale,primary_exchange,type,active,currency_name,cik,composite_figi,share_class_figi,last_updated_utc
0,A,Agilent Technologies Inc.,stocks,us,XNYS,CS,True,usd,0001090872,BBG000C2V3D6,BBG001SCTQY4,2026-03-27T06:08:03.745778795Z
1,AA,Alcoa Corporation,stocks,us,XNYS,CS,True,usd,0001675149,BBG00B3T3HD3,BBG00B3T3HF1,2026-03-27T06:08:03.745779346Z
2,AAMI,Acadian Asset Management Inc.,stocks,us,XNYS,CS,True,usd,0001748824,BBG00P2HLNY3,BBG00P2HLNZ2,2026-03-27T06:08:03.74578155Z
3,AAP,ADVANCE AUTO PARTS INC,stocks,us,XNYS,CS,True,usd,0001158449,BBG000F7RCJ1,BBG001SD2SB2,2026-03-27T06:08:03.745781891Z
4,AAT,"AMERICAN ASSETS TRUST, INC.",stocks,us,XNYS,CS,True,usd,0001500217,BBG00161BCR0,BBG001TCBJS5,2026-03-27T06:08:03.745783304Z


## 1. Basic counts

In [3]:
# Shape and column overview
print('Shape:', df.shape)
print()
# Non-null counts + dtype for each column
print(df.info())

Shape: (5028, 12)

<class 'pandas.DataFrame'>
RangeIndex: 5028 entries, 0 to 5027
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   ticker            5028 non-null   str  
 1   name              5028 non-null   str  
 2   market            5028 non-null   str  
 3   locale            5028 non-null   str  
 4   primary_exchange  5028 non-null   str  
 5   type              5028 non-null   str  
 6   active            5028 non-null   bool 
 7   currency_name     5028 non-null   str  
 8   cik               5021 non-null   str  
 9   composite_figi    3999 non-null   str  
 10  share_class_figi  3998 non-null   str  
 11  last_updated_utc  5028 non-null   str  
dtypes: bool(1), str(11)
memory usage: 981.3 KB
None


In [4]:
# Null counts per column
df.isnull().sum().sort_values(ascending=False)

share_class_figi    1030
composite_figi      1029
cik                    7
ticker                 0
name                   0
market                 0
locale                 0
primary_exchange       0
type                   0
active                 0
currency_name          0
last_updated_utc       0
dtype: int64

In [5]:
# Unique value counts for categorical columns
for col in ['market', 'locale', 'primary_exchange', 'type', 'active', 'currency_name']:
    if col in df.columns:
        print(f'\n── {col} ──')
        print(df[col].value_counts())


── market ──
market
stocks    5028
Name: count, dtype: int64

── locale ──
locale
us    5028
Name: count, dtype: int64

── primary_exchange ──
primary_exchange
XNAS    3291
XNYS    1737
Name: count, dtype: int64

── type ──
type
CS    5028
Name: count, dtype: int64

── active ──
active
True    5028
Name: count, dtype: int64

── currency_name ──
currency_name
usd    5028
Name: count, dtype: int64


## 2. What makes a good dim table?

In [6]:
# Keep only useful columns for the dim table
keep = ['ticker', 'name', 'primary_exchange', 'type', 'active', 'cik', 'currency_name']
keep = [c for c in keep if c in df.columns]
dim = df[keep].drop_duplicates(subset='ticker').reset_index(drop=True)
print(f'Dim table: {dim.shape}')
dim.head(10)

Dim table: (5028, 7)


,ticker,name,primary_exchange,type,active,cik,currency_name
0,A,Agilent Technologies Inc.,XNYS,CS,True,0001090872,usd
1,AA,Alcoa Corporation,XNYS,CS,True,0001675149,usd
2,AAMI,Acadian Asset Management Inc.,XNYS,CS,True,0001748824,usd
3,AAP,ADVANCE AUTO PARTS INC,XNYS,CS,True,0001158449,usd
4,AAT,"AMERICAN ASSETS TRUST, INC.",XNYS,CS,True,0001500217,usd
5,AAUC,Allied Gold Corporation,XNYS,CS,True,0001993344,usd
6,AB,"AllianceBernstein Holding, L.P.",XNYS,CS,True,0000825313,usd
7,ABBV,ABBVIE INC.,XNYS,CS,True,0001551152,usd
8,ABCB,Ameris Bancorp,XNYS,CS,True,0000351569,usd
9,ABG,"Asbury Automotive Group, Inc.",XNYS,CS,True,0001144980,usd


In [7]:
# How many tickers have a CIK (linkable to SEC filings)?
has_cik = dim['cik'].notna().sum()
print(f'{has_cik}/{len(dim)} tickers have a CIK ({has_cik/len(dim)*100:.1f}%)')

5021/5028 tickers have a CIK (99.9%)


In [8]:
# Sample a few rows to sanity check
dim.sample(10)

,ticker,name,primary_exchange,type,active,cik,currency_name
2621,DSGN,"Design Therapeutics, Inc. Common Stock",XNAS,CS,True,0001807120,usd
74,ALLE,Allegion Public Limited Company,XNYS,CS,True,0001579241,usd
1650,VVX,"V2X, Inc.",XNYS,CS,True,0001601548,usd
4606,TBHC,"The Brand House Collective, Inc. Common Stock",XNAS,CS,True,0001056285,usd
2396,CMCO,Columbus McKinnon Corp/NY,XNAS,CS,True,0001005229,usd
1667,WEX,WEX Inc.,XNYS,CS,True,0001309108,usd
4205,RDI,"Reading International, Inc Class A Common Stock",XNAS,CS,True,0000716634,usd
943,LOB,"Live Oak Bancshares, Inc.",XNYS,CS,True,0001462120,usd
4854,VRSN,VeriSign Inc,XNAS,CS,True,0001014473,usd
1573,UA,"Under Armour, Inc. Class C Common Stock, $0.00...",XNYS,CS,True,0001336917,usd
